# Interim: combining the raw files for mid cap

- This is the storage step of our pipeline (raw, then interim, then clean, then dashboards). It takes the files saved in raw/ and puts them into one NAV table and one TER table. Both are saved in interim/ as CSV and as Parquet.We will tell the reason for the formats we choose. Raw files are not touched, so if something here goes wrong we can delete interim/ and build it again.

**Why we redid this notebook.**
- Our raw files just stacked every NAV file. That gave 12,120,812 rows and a 1.5 GB text files, and the first rows we looked at were `quant Multi Cap Fund`. AMFI sends all open ended schemes of a fund house in each file and our project is only about mid cap, so most of that table was not ours. The data is related to our project (`Equity Scheme - Mid Cap Fund` and `Equity Schemes - Mid Cap Fund`). `Large & Mid Cap` is dropped.

## How the data is stored
- The raw folder contains four sub folders out of which each are ter,nav,robots and check.The most important data is in ter and nav.
- In nav there is each file for one mutual fund scheme and inside each mf{id} type there contains all the data in a set of 3 months
- In ter folder there are folders for all years and each year contains the corresponding json files
- **One table instead of 1,088 files.** The later notebooks should read one file and not loop over folders. The fund house is not lost, it moves into the `MF_ID` and `AMC` columns.

- **raw** is the data exactly as it was downloaded. We never edit it.
- **interim** puts the raw files into tables. Nothing is cleaned here, we only combine and store.
- **clean** (next notebook) filters the plans, changes the types and matches NAV with TER.
- Raw files are not touched here. If something goes wrong we can delete the interim/ folder and run this notebook again.

**Why we did this step again.** The raw NAV files have all the open ended schemes of a fund house, not only mid cap. If we stack all of them it becomes about 12 million rows and a 1.5 GB text file and the first rows are funds like quant Multi Cap Fund. So here we read the files line by line and keep only the mid cap rows.

- **One table instead of 1,088 files.** The next notebooks should read one file and not loop over folders. The fund house is not lost, it moves into the `MF_ID` and `AMC` columns.
- **Why not keep the raw formats?** The raw NAV file has the category in a heading line and the fund house name on its own line, so it is not a table. The raw TER JSON repeats every column name in every record, which makes it big. Once the rows are in one table with proper columns, a table format is the right home for it.

###  File formats: CSV and Parquet

We save every table in two formats.

- CSV is the copy anyone can open (Excel, R, a text editor) and the one to look at when something seems off. It is comma separated, the semicolons were only in the raw AMFI files.
- Parquet is the working file. It is compressed so it is small, it loads fast and it remembers the column types. For example `TER_Date` stays a real date in the Parquet but becomes plain text in the CSV. It also opens in R and Spark.
- Parquet needs the pyarrow library, which pandas uses to write it.

- The below are the tradeoffs between selecting csv and parquet files
| | CSV | Parquet |
|---|---|---|
| Who can open it | anyone, Excel and text editors too ||
| Size | big | small, it is compressed |
| Column types | lost, dates become text | remembered |
| Load speed | slower | faster |
| Our use | the copy to look at | the file the next notebook reads |

**Why not pickle or SQLite?**
- A pickle only works in Python, can break with a different pandas version and is not safe to open if somebody else made it. We used it in the first version and dropped it.
- SQLite is useful to join tables with SQL, but we only read full tables and the NAV and TER matching is done in pandas in the clean notebook, so plain files are enough.

**Everything is still text here.** NAV values, dates and TER values are not changed to numbers in this notebook. The interim table should be a faithful copy of the raw rows and changing the types is a cleaning decision.

### Imports and Setup

In [1]:
!pip install pyarrow

In [2]:
import os
import json
import glob
import time
from datetime import datetime
import pandas as pd

try:
    import pyarrow
except ImportError:
    raise ImportError("parquet needs pyarrow, run: pip install pyarrow")
for directory in ["interim", "interim/nav", "interim/ter"]:
    os.makedirs(directory, exist_ok=True)
TODAY = "2026-10-26"
MIDCAP_TER_CATS = [76, 17]
print("Raw data was collected on:", TODAY)

Raw data was collected on: 2026-10-26


## NAV table

### What a raw NAV file has 
The category is not a column. A raw file is built like this:
```
Scheme Code;NAV Name;Plan;Option;...;Net Asset Value;Date          <- header line
Open Ended Schemes(Equity Scheme - Mid Cap Fund)                    <- heading line = the category
<fund house name>                                                   <- fund house on its own line
<code>;<scheme name>;<plan>;<option>;<isin>;<isin>;<nav>;<date>    <- data rows, until the next heading
```

In [3]:
# THis code prints how actually the nav files are
mf_sample = 3
sample_file = sorted(glob.glob(f"raw/nav/mf{mf_sample}/*.txt"))[-1]
print(sample_file)
with open(sample_file, encoding="utf-8", errors="replace") as f:
    for i in range(10):
        print(repr(f.readline().strip()[:110]))

raw/nav/mf3/amfi_nav_mf3_30-Nov-2024_27-Feb-2025_2026-10-22.txt
'Scheme Code;NAV Name;Plan;Option;ISIN Div Payout/ISIN Growth;ISIN Div Reinvestment;Net Asset Value;Date'
''
'Open Ended Schemes ( Equity Scheme - Multi Cap Fund )'
''
''
'Aditya Birla Sun Life Mutual Fund'
'148921;Aditya Birla Sun Life Multi-Cap Fund-Direct Growth;Direct Plan;GROWTH;INF209KB1Y49;;20.67;02-Dec-2024'
'148921;Aditya Birla Sun Life Multi-Cap Fund-Direct Growth;Direct Plan;GROWTH;INF209KB1Y49;;20.85;03-Dec-2024'
'148921;Aditya Birla Sun Life Multi-Cap Fund-Direct Growth;Direct Plan;GROWTH;INF209KB1Y49;;20.94;04-Dec-2024'
'148921;Aditya Birla Sun Life Multi-Cap Fund-Direct Growth;Direct Plan;GROWTH;INF209KB1Y49;;20.98;05-Dec-2024'


###  Reading one file and keeping only mid cap
We want to extract the Equity scheme/s midcap in all files so we create a function that does this
The function read_nav_file() goes through a file line by line and does this:

1. Remembers the latest heading line and the latest fund house line.
2. A line whose first field is only digits is a data row. It must have exactly 8 fields, otherwise it is counted as a bad row.
3. The row is kept only if the heading is mid cap. Then we add three columns to it: Category (the heading), AMC (the fund house name) and MF_ID (from the folder name `mf<id>`).

In [4]:
NAV_COLUMNS = ["Scheme Code", "NAV Name", "Plan", "Option",
               "ISIN Div Payout/ISIN Growth", "ISIN Div Reinvestment",
               "Net Asset Value", "Date"]
ALL_COLUMNS = NAV_COLUMNS + ["Category", "AMC", "MF_ID"]

def is_midcap(heading):
    text = heading.lower().replace("-", " ")
    return ("mid cap" in text or "midcap" in text) and "large" not in text

def read_nav_file(path, mf_id):
    kept_rows = []
    heading = ""
    amc = ""
    all_rows = 0
    bad_rows = 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if line == "" or line.startswith("Scheme Code"):
                continue
            fields = line.split(";")
            if fields[0].isdigit():
                # a data row
                if len(fields) != len(NAV_COLUMNS):
                    bad_rows += 1
                    continue
                all_rows += 1
                if is_midcap(heading):
                    kept_rows.append(fields + [heading, amc, mf_id])
            elif line.startswith(("Open Ended", "Close Ended", "Interval")):
                heading = line
            else:
                amc = line
    return kept_rows, all_rows, bad_rows

Testing it on the file we used previously

In [5]:
rows, all_rows, bad_rows = read_nav_file(sample_file, mf_sample)
one = pd.DataFrame(rows, columns=ALL_COLUMNS)
print("data rows seen:", all_rows, "| mid cap rows kept:", len(one),
      "| bad rows:", bad_rows)
print(one["Category"].value_counts())
one.head()

data rows seen: 26679 | mid cap rows kept: 248 | bad rows: 0
Category
Open Ended Schemes ( Equity Scheme - Mid Cap Fund )    248
Name: count, dtype: int64


,Scheme Code,NAV Name,Plan,Option,ISIN Div Payout/ISIN Growth,ISIN Div Reinvestment,Net Asset Value,Date,Category,AMC,MF_ID
0,119620,Aditya Birla Sun Life Midcap Fund - Growth - D...,Direct Plan,GROWTH,INF209K01Q30,,864.72,02-Dec-2024,Open Ended Schemes ( Equity Scheme - Mid Cap F...,Aditya Birla Sun Life Mutual Fund,3
1,119620,Aditya Birla Sun Life Midcap Fund - Growth - D...,Direct Plan,GROWTH,INF209K01Q30,,874.86,03-Dec-2024,Open Ended Schemes ( Equity Scheme - Mid Cap F...,Aditya Birla Sun Life Mutual Fund,3
2,119620,Aditya Birla Sun Life Midcap Fund - Growth - D...,Direct Plan,GROWTH,INF209K01Q30,,878.72,04-Dec-2024,Open Ended Schemes ( Equity Scheme - Mid Cap F...,Aditya Birla Sun Life Mutual Fund,3
3,119620,Aditya Birla Sun Life Midcap Fund - Growth - D...,Direct Plan,GROWTH,INF209K01Q30,,878.55,05-Dec-2024,Open Ended Schemes ( Equity Scheme - Mid Cap F...,Aditya Birla Sun Life Mutual Fund,3
4,119620,Aditya Birla Sun Life Midcap Fund - Growth - D...,Direct Plan,GROWTH,INF209K01Q30,,878.87,06-Dec-2024,Open Ended Schemes ( Equity Scheme - Mid Cap F...,Aditya Birla Sun Life Mutual Fund,3


### Combining all NAV files

The same function is used on every file in raw/nav/mf*/.
In the raw data we saw: 34 fund houses, 1,088 files and 67 of them are empty (a fund that had nothing in that 90 day chunk, for example a fund that started late). Empty files give no rows, which is fine.

In [6]:
nav_paths = sorted(glob.glob("raw/nav/mf*/*.txt"))
print("raw NAV files:", len(nav_paths), "(expected 1,088)")
if len(nav_paths) == 0:
    raise FileNotFoundError("no NAV files in raw/nav/mf*/, run the raw notebook first")
frames = []
total_rows = 0
total_bad = 0
for i, path in enumerate(nav_paths, 1):
    mf_id = int(os.path.basename(os.path.dirname(path)).replace("mf", ""))
    rows, all_rows, bad_rows = read_nav_file(path, mf_id)
    if rows:
        frames.append(pd.DataFrame(rows, columns=ALL_COLUMNS))
    total_rows += all_rows
    total_bad += bad_rows
    if i % 200 == 0:
        print(i, "of", len(nav_paths), "files read")
nav_mid = pd.concat(frames, ignore_index=True)
print("data rows in raw files (all categories):", total_rows)
print("same as the first version (12,120,812)?", total_rows == 12120812)
print("mid cap rows kept:", len(nav_mid))
print("rows with the wrong number of fields:", total_bad)
print("table shape:", nav_mid.shape)

raw NAV files: 1088 (expected 1,088)
200 of 1088 files read
400 of 1088 files read
600 of 1088 files read
800 of 1088 files read
1000 of 1088 files read
data rows in raw files (all categories): 11957321
same as the first version (12,120,812)? False
mid cap rows kept: 206239
rows with the wrong number of fields: 0
table shape: (206239, 11)


Checking that only the two mid cap headings are in the table.

In [7]:
print(nav_mid["Category"].value_counts())
assert nav_mid["Category"].map(is_midcap).all()

Category
Open Ended Schemes ( Equity Scheme - Mid Cap Fund )     146051
Open Ended Schemes ( Equity Schemes - Mid Cap Fund )     60188
Name: count, dtype: int64


### Plans and options

We still keep every plan and option here, this is just to see what is inside. Direct and Regular, Growth only is done in the clean notebook

### Coverage per fund house

Number of schemes, rows and the first and last NAV date for each fund house. We expect 34 fund houses. A fund house whose first date is late is a fund that launched late. The last date should not be later than 30-Sep-2026.

In [8]:
nav_dates = pd.to_datetime(nav_mid["Date"], format="%d-%b-%Y", errors="coerce")
print("dates that did not parse:", nav_dates.isna().sum())

cover = nav_mid.assign(date=nav_dates).groupby("MF_ID").agg(
    schemes=("Scheme Code", "nunique"),
    rows=("Scheme Code", "size"),
    first_nav=("date", "min"),
    last_nav=("date", "max"))
print("fund houses:", len(cover), "(should be 34)")
print(cover)

dates that did not parse: 0
fund houses: 34 (should be 34)
       schemes   rows  first_nav   last_nav
MF_ID                                      
3            4   7644 2019-01-01 2026-09-30
4            8   7648 2019-01-01 2026-09-30
6            4   7644 2019-01-01 2026-09-30
9            4   7644 2019-01-01 2026-09-30
13           4   7648 2019-01-01 2026-09-30
16           4   3796 2022-11-25 2026-09-30
17           4   7644 2019-01-01 2026-09-30
18           4   3124 2023-07-31 2026-09-30
20           4   7644 2019-01-01 2026-09-30
21           7  13377 2019-01-01 2026-09-30
22           4   7680 2019-01-01 2026-09-30
25           4   7644 2019-01-01 2026-09-30
26           4   7656 2019-01-01 2026-09-30
27           4   7644 2019-01-01 2026-09-30
28           4   7644 2019-01-01 2026-09-30
32           4   3768 2022-12-06 2026-09-30
33           6   9126 2019-01-01 2026-09-30
37           8   4936 2021-09-29 2026-09-30
42           4   7644 2019-01-01 2026-09-30
45           4   

In [9]:
by_year = nav_mid.assign(year=nav_dates.dt.year).groupby("year").agg(
    rows=("Scheme Code", "size"),
    schemes=("Scheme Code", "nunique"))
print(by_year)

       rows  schemes
year                
2019  21168       89
2020  23014       93
2021  24027      101
2022  25313      123
2023  28468      119
2024  29306      119
2025  30684      127
2026  24259      135


In [10]:
def save_table(df, stem):
    csv_path = stem + ".csv"
    parquet_path = stem + ".parquet"
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)
    for path in [csv_path, parquet_path]:
        print(path, round(os.path.getsize(path) / 1024**2, 1), "MB")
    return csv_path, parquet_path

nav_csv, nav_parquet = save_table(nav_mid, "interim/nav/nav_midcap_combined")

interim/nav/nav_midcap_combined.csv 39.1 MB
interim/nav/nav_midcap_combined.parquet 1.5 MB


In [11]:
start = time.time()
pd.read_csv(nav_csv, dtype=str)
csv_seconds = round(time.time() - start, 2)

start = time.time()
pd.read_parquet(nav_parquet)
parquet_seconds = round(time.time() - start, 2)

print("NAV load time | csv:", csv_seconds, "s | parquet:", parquet_seconds, "s")
print("NAV file size | csv:", round(os.path.getsize(nav_csv) / 1024**2, 1), "MB | parquet:",
      round(os.path.getsize(nav_parquet) / 1024**2, 1), "MB")

NAV load time | csv: 1.17 s | parquet: 0.28 s
NAV file size | csv: 39.1 MB | parquet: 1.5 MB


###  Reading all the TER pages

- The files are in raw/ter/<year>/ and the name looks like amfi_ter_midcap_cat76_08-2026_p1_2026-10-22.json.
- From the name we take the category id (cat76), the month and the page number.
- From inside the file we take the records and the numbers in meta (pageCount and total), which we need to check that no page is missing.
- We add the category id as a column called strCat.

In [12]:
ter_paths = sorted(glob.glob("raw/ter/*/amfi_ter_midcap_cat*_p*_*.json"))
print("TER JSON files:", len(ter_paths))
ter_rows = []
page_list = []
for path in ter_paths:
    parts = os.path.basename(path).split("_")
    cat = int(parts[3].replace("cat", ""))
    month = parts[4]
    page = int(parts[5].replace("p", ""))
    with open(path, encoding="utf-8") as f:
        page_data = json.load(f)
    page_list.append((cat, month, page, page_data["meta"]["pageCount"],
                      page_data["meta"]["total"], len(page_data["data"])))
    for record in page_data["data"]:
        record["strCat"] = cat
        ter_rows.append(record)
print("records read:", len(ter_rows), "(the raw notebook counted 70,019)")

TER JSON files: 795
records read: 70019 (the raw notebook counted 70,019)


Checking all files are present or not in the page_list

In [13]:
page_df = pd.DataFrame(page_list, columns=["cat", "month", "page", "page_count", "total", "records"])
check = page_df.groupby(["cat", "month"]).agg(
    pages_found=("page", "nunique"),
    page_count=("page_count", "first"),
    total=("total", "first"),
    records=("records", "sum"))
incomplete = check[(check["pages_found"] != check["page_count"]) | (check["records"] != check["total"])]
print("category-months checked:", len(check), "(2 categories x 93 months = 186)")
print("incomplete:", len(incomplete))
print(incomplete)

category-months checked: 186 (2 categories x 93 months = 186)
incomplete: 0
Empty DataFrame
Columns: [pages_found, page_count, total, records]
Index: []


### Combining

- Making All the records go into one table.
- TER_Date is changed to a real date (the time part is always 00:00) and the table is sorted by it.
- MF_ID is made a whole number, same as in the NAV table.

In [14]:
ter_df = pd.DataFrame(ter_rows)

ter_df["TER_Date"] = pd.to_datetime(ter_df["TER_Date"], utc=True, errors="coerce").dt.tz_localize(None)
ter_df["MF_ID"] = ter_df["MF_ID"].astype(int)
for col in ter_df.columns:
    if col not in ["TER_Date", "MF_ID", "strCat"]:
        ter_df[col] = ter_df[col].astype("string")

ter_df = ter_df.sort_values(["TER_Date", "MF_ID", "Scheme_Name"]).reset_index(drop=True)

print("TER records:", len(ter_df))
print("TER_Date that did not parse:", ter_df["TER_Date"].isna().sum())
print("date range:", ter_df["TER_Date"].min().date(), "to", ter_df["TER_Date"].max().date())
print("repeated rows:", ter_df.duplicated().sum())
print("repeated scheme-days:", ter_df.duplicated(["MF_ID", "Scheme_Name", "SchemeCat_Desc", "TER_Date"]).sum())

TER records: 70019
TER_Date that did not parse: 0
date range: 2019-01-01 to 2026-09-30
repeated rows: 0
repeated scheme-days: 0


### Saving TER

Same function, so we get the CSV and the Parquet. Here the CSV keeps `TER_Date` as text while the Parquet keeps it as a date.

In [15]:
ter_csv, ter_parquet = save_table(ter_df, "interim/ter/ter_combined")

interim/ter/ter_combined.csv 12.5 MB
interim/ter/ter_combined.parquet 0.3 MB


## interim at present

In [17]:
for path in sorted(glob.glob("interim/*/*")):
    print(path, round(os.path.getsize(path) / 1024**2, 1), "MB")

interim/nav/nav_midcap_combined.csv 39.1 MB
interim/nav/nav_midcap_combined.parquet 1.5 MB
interim/ter/ter_combined.csv 12.5 MB
interim/ter/ter_combined.parquet 0.3 MB


- interim/nav/nav_midcap_combined.csv and .parquet: NAV of the mid cap categories only, all plans and options, with Category, AMC and MF_ID columns added.
- interim/ter/ter_combined.csv and .parquet: TER for both mid cap categories, with strCat added.